# TrialScreen: RESUME after training and evaluation

Reuses the adapter and results from your earlier run, then runs only what is left:
Task 3 (KV cache), Task 4 (attention), Task 5 (speculative), report assets, adapter upload, zip.

**Before running:**
1. Right panel → **Add Input** → **Notebooks / Your Work** → add the earlier notebook version that has the
   trained adapter and the evaluation results (the run whose output contains
   `trialscreen/checkpoints/.../best_adapter` and `trialscreen/results/metrics_finetuned_test.json`).
2. Accelerator **GPU T4**, Internet **On**, secret **HF_TOKEN** attached.
3. **Save Version → Save & Run All (Commit)**. About 30 to 45 minutes.

In [ ]:
# 0. Checks
import os, subprocess, sys, time, glob, shutil
WORK = "/kaggle/working"
net = subprocess.run(["curl", "-sI", "--max-time", "15", "https://github.com"], capture_output=True, text=True).stdout.split("\n")[0]
assert net.startswith("HTTP"), "No internet: Settings > Internet On."
import torch
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4."
print(torch.cuda.get_device_name(0), "capability", torch.cuda.get_device_capability())
T0 = time.time()

In [ ]:
# 1. Fresh code
REPO = "https://github.com/eman-huda/trialscreen.git"
ROOT = f"{WORK}/trialscreen"
subprocess.run(["rm", "-rf", ROOT])
subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
os.chdir(ROOT)
src = open("src/trialscreen/decoding.py").read() + open("src/trialscreen/modeling.py").read()
assert "logits_to_keep" in src and "get_device_capability" in src, \
    "Repo is missing the latest fixes (trialscreen_update2.zip). Push them from Codespaces, then rerun."
print(subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install
!pip install -q "transformers==5.22.2" "peft==0.21.2" "accelerate==1.15.0" scikit-learn matplotlib pytest
!pip install -q -e .
!pip uninstall -y -q torchao

In [ ]:
# 3. Bring back the trained adapter and earlier results from the attached input
ad = sorted(glob.glob("/kaggle/input/**/best_adapter/adapter_model.safetensors", recursive=True))
assert ad, "No adapter found under /kaggle/input. Add the earlier notebook version as an input (see top cell)."
src_adapter = os.path.dirname(ad[0])
dst_adapter = f"{ROOT}/checkpoints/qwen2.5-1.5b-trialscreen-lora/best_adapter"
os.makedirs(os.path.dirname(dst_adapter), exist_ok=True)
shutil.copytree(src_adapter, dst_adapter, dirs_exist_ok=True)
print("adapter from", src_adapter)
for f in os.listdir(os.path.dirname(src_adapter)):           # history.json / train_meta.json next to it
    p = os.path.join(os.path.dirname(src_adapter), f)
    if f.endswith(".json"):
        shutil.copy(p, os.path.dirname(dst_adapter))
os.makedirs(f"{ROOT}/results", exist_ok=True)
keep = ("train_", "metrics_", "predictions_", "attention.json", "pytest")
copied = []
for p in glob.glob("/kaggle/input/**/results/*", recursive=True):
    n = os.path.basename(p)
    if os.path.isfile(p) and n.startswith(keep) and not os.path.exists(f"{ROOT}/results/{n}"):
        shutil.copy(p, f"{ROOT}/results/{n}"); copied.append(n)
print("results restored:", sorted(copied))
need = ["metrics_base_test.json", "metrics_finetuned_test.json", "metrics_base_validation.json",
        "metrics_finetuned_validation.json", "predictions_finetuned_test.jsonl", "train_history.json", "train_meta.json"]
missing = [n for n in need if not os.path.exists(f"{ROOT}/results/{n}")]
print("MISSING (rerun 02/03 later if needed):", missing) if missing else print("all training/evaluation results present")

## Task 3: KV-cache benchmark (fp16 on T4)

In [ ]:
!python -u scripts/04_bench_kv.py

## Task 4: attention

In [ ]:
!python -u scripts/05_attention.py

## Task 5: speculative decoding

In [ ]:
!python -u scripts/06_bench_speculative.py

## Report assets

In [ ]:
!python scripts/07_make_report_assets.py
!ls report/figures report/tables results

## Upload the adapter to Hugging Face

In [ ]:
HF_REPO = "eman-huda/qwen2.5-1.5b-trialscreen-lora"   # change if your HF username differs
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    r = subprocess.run([sys.executable, "scripts/08_push_adapter.py", "--repo", HF_REPO], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
except Exception as e:
    print("Adapter NOT uploaded:", e)

## Package everything

In [ ]:
out = f"{WORK}/trialscreen_outputs.zip"
!cd {ROOT} && zip -qr {out} results report data/splits checkpoints
!ls -lh {out}
print(f"total {(time.time() - T0) / 60:.0f} min")